In [39]:
# Import packages

%matplotlib qt
from matplotlib.pyplot import plot, show, title

import scipy
import scipy.signal

import numpy
from numpy import loadtxt
from numpy import savetxt
import numpy as np

import biosppy
from biosppy.signals import ecg

from opensignalsreader import OpenSignalsReader

import pyhrv.tools as tools
from pyhrv.hrv import hrv

import csv

# ECG file upload

fs - Recording ECG frequency. Swell - 2048 Hz; Wesad - 700 Hz

In [40]:
#/media/eduarda/Seagate/RawFilteredSwellData/S1_stress_I.csv
#/media/eduarda/Seagate/RawWesad/S2_baseline_ECG_values.csv
ecg = loadtxt("/media/eduarda/Seagate/RawFilteredSwellData/S1_neutral.csv", delimiter=',')

#Swell - 2048Hz; Wesad - 700Hz
fs = 2048

# Filtering ButterWorth Filter (Wesad: 0.5-40Hz)

In [41]:
def bandpassfilter(signal, fs):
    lowcut=0.5 #8
    highcut=40 #55
    
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    
    order = 2
    
    b, a = scipy.signal.butter(order, [low,high], 'bandpass', analog=False)
    y = scipy.signal.filtfilt(b,a, signal, axis=0)
    
    return(y)

In [42]:
#--------------WESAD--------------------

#filtered_ecg = bandpassfilter(ecg, fs)

#--------------SWELL--------------------

filtered_ecg=ecg

In [43]:
#number of samples in the .csv file, i.e. number of data points
#print(filtered_ecg.size)

In [44]:
#S1_stress_I (c3): duration 3367000 (ms)
#S2_baseline: duration 1518312 (ms)

ecg_duration = 3525 #s
samples = 7219696 # = filtered_ecg.size
time = ecg_duration / samples # time between consecutive data points in the .csv file

time_frame = 30

hrv_csv_name = 'test.csv' 

## Plotting ECG: Original and Filtered ; Saving filtered file optional

In [45]:
#title("Original signal")
#plot(ecg)
#show()

#title("Filtered signal")
#plot(data)
#show()

#from numpy import savetxt
#savetxt( "S2_stress_ECG_values_FILTERED.csv", data, delimiter=',')

# Splitting ECG in 30s time frames

In [46]:
batch = time_frame // time #number of samples that equal 30s

number_of_arrays = samples // batch # number of total splits of ecg file

split_values = np.array_split(filtered_ecg, number_of_arrays)

In [47]:
#title("Filtered signal")
#plot(split_values[0])
#show()

nao perco o tempo pq os pontos continuam a tar separados pelo mm tempo, soa valores relativos os do rr
de facto agora deixo é de tar os valores de tempo em si, mas é como se tivesse a fazer mini ecgs.

# Detecting R Peaks with Biospy (Hamilton)

https://biosppy.readthedocs.io/en/stable/biosppy.signals.html#biosppy-signals-ecg

In [48]:
def rPeakHamiltonTempo(filtered_ecg, fs, time):
    #Returns R-peak location indices (sample number, IT IS NOT TIME)
    rPeakLocation = biosppy.signals.ecg.hamilton_segmenter(signal=filtered_ecg, sampling_rate=fs)
    
    #Obtaining R-peak values in time (s): R-peak sample number -> R-peak time value
    tt = []
    
    for i in rPeakLocation:
        xx = i * time
        tt.append(xx)
        i = i + 1
    
    print (tt)
    
    return tt

# Detecting R Peaks with default parameters

https://biosppy.readthedocs.io/en/stable/biosppy.signals.html#biosppy-signals-ecg

In [49]:
def rPeak(filtered_ecg, fs, time):
    #Returns R-peak location indices (sample number, IT IS NOT TIME)
    rPeakLocation = biosppy.signals.ecg.ecg(filtered_ecg, sampling_rate=fs, show = False)[2]
    
    #Obtaining R-peak values in time (s): R-peak sample number -> R-peak time value
    tt = []
    
    for i in rPeakLocation:
        xx = i * time
        tt.append(xx)
        i = i + 1
    
    print (tt)
    
    return tt

## The variable `rPeakss` returns the time in which the R-peaks occured within a 30s time frame
## The list `rPeaks` contains all rPeaks of the ecg

In [50]:
#R-peak Hamilton
#rPeaks = rPeakHamiltonTempo(filtered_ecg, fs, time)

#R-peaks default parameters
#rPeaks = rPeak(filtered_ecg, fs, time)


#l is a list that contains x numbers of lists, x =number_of_arrays. 
#l[0] contains the time in which the rPeaks for the first 30s of measurmente occured.

rPeaks=[]
for n in range (len(split_values)):
    #R-peak Hamilton
    #rPeakss = rPeakHamiltonTempo(split_values[n], fs, time)

    #R-peaks default parameters
    rPeakss = rPeak(split_values[n], fs, time)
    
    
    rPeaks.append(rPeakss)


[0.40085136548685707, 1.2699322796970953, 1.884147892099612, 2.7346753935345753, 3.5441900877820895, 4.346869314165029, 5.145154311206455, 5.943927555952494, 6.763207204292258, 8.732798444699057, 10.12381615514005, 10.69360122642283, 11.230673701496572, 11.785323093936364, 12.387820761428182, 13.194894217152633, 13.98536725092026, 14.749474908638813, 16.082391142230918, 16.917294717118285, 17.74292158561801, 18.554877518388587, 19.33851508429164, 20.113852439216277, 20.858918436454942, 21.582013286986044, 22.28997245867416, 22.974007492836265, 23.664389747158328, 24.325477139203645, 25.38595115362198, 25.76336662928744, 26.382952966440694, 27.00498054211701, 27.6221256407472, 28.229994032989755, 28.81833251704781, 29.402276771764353]
[0.4345404571051191, 1.0111609962524737, 1.5828990583537035, 2.1565901112733834, 2.730281164193063, 3.2990897400666177, 4.147176002978519, 4.699872404599861, 5.249639319993529, 5.806729950956384, 6.365773572737688, 6.874039433239294, 7.484837311709524, 8.0

[0.6605991443407035, 1.5799695721260285, 2.5310761007111657, 3.5559080326927894, 4.571951505991388, 5.518663805235013, 6.421433811063513, 7.323227321482788, 8.265057143680288, 9.238623066677599, 10.179476393465874, 11.109100023048063, 11.9508390658, 12.759865512342902, 13.619181472460891, 14.575170478092152, 15.480381722443715, 16.39779915941059, 17.37478281634019, 18.381061335546537, 19.333632607245512, 20.25349128273545, 21.188973884772988, 22.20355261495775, 23.256214666102284, 24.27176989169627, 25.249730044035097, 26.218413489986283, 27.186608688232855, 28.14259769386412, 29.095657213267707]
[1.0004195467509989, 2.0101157998896353, 2.9929584292745846, 3.978730544887209, 4.99721525670887, 6.024000179508944, 7.01807250609998, 7.953066860432905, 8.897337921153467, 9.873345082673842, 10.823475115849753, 11.740892552816629, 12.68516361353719, 13.698277600608114, 14.755333881094163, 15.812878409284822, 16.870911185180095, 17.933338190416883, 18.973305801241494, 19.94980121046648, 20.889

[0.4189165305575193, 1.2728617659247703, 2.1077653408121337, 2.92558024603806, 3.6999411055534748, 4.44500710279214, 5.177866907415492, 5.8755728773067455, 6.581090810471799, 7.311021128867476, 8.094170447065915, 8.93298000359018, 9.73224149604083, 10.487560695076358, 11.248738866567235, 12.026029212310325, 12.843844117536252, 13.695836362085052, 14.554175826793815, 15.43058045657324, 16.30112611389732, 17.06376902850203, 17.810788016559147, 18.550483289047072, 19.29750227710419, 20.10848171446554, 20.984398096540353, 21.89986254268878, 22.84999257586469, 23.784986930197615, 24.726816752395116, 25.64423418936199, 26.44886640656338, 27.197350137734333, 27.932162933176134, 28.63279838929506, 29.36272870769074]
[0.9603832349727744, 1.8861008829180619, 2.717586723873138, 3.531007399757552, 4.3395455985958415, 5.172984430369367, 6.055736280308755, 6.996589607097031, 7.95599634666058, 8.907591122950329, 9.831844027781779, 10.652100171530767, 11.423531544818507, 12.194962918106247, 13.0035011

[1.0189729595262738, 1.8934245984872493, 2.6912213478240634, 3.4787648953640153, 4.2731439107685425, 5.106094494837456, 5.992264078709131, 6.932140910088181, 7.877876713922581, 8.845583664464543, 9.789854725185105, 10.712154639198104, 11.587582773568306, 12.458616678596993, 13.338439042308707, 14.229979350931119, 15.155208751171795, 16.106803527461544, 17.054492322114395, 17.95286809860138, 18.837084691654606, 19.710071587501744, 20.582570235644273, 21.453115892968345, 22.310478862267885, 23.126340776675363, 23.936343718627487, 24.760017596308764, 25.62861026281439, 26.530403773233665, 27.41608510940073, 28.26221838149418, 29.11030464440608]
[0.7616664191954897, 1.5692281226245537, 2.429520578151767, 3.197045969802607, 3.9772658017733713, 4.784339257497822, 5.663185125800311, 6.6045267002931975, 7.588345825087372, 8.566305977426197, 9.603344102023133, 10.738031767542568, 11.916173478772514, 12.867280007357651, 13.768097022367701, 14.664031560331626, 15.505770603083565, 16.3265149945371

[0.5717380621012298, 1.443748462539143, 2.280116780540344, 3.086701988560183, 3.902075655263047, 4.743326450310374, 5.626078300249762, 6.553748939013499, 7.493625770392549, 8.423737647679348, 9.332366625963198, 10.2312306501548, 11.107635279934225, 11.973298460212176, 12.876556713745288, 13.859399343130237, 14.863236623813524, 15.860726684336848, 16.831851368811098, 17.790769860670032, 18.746758866301295, 19.674429505065035, 20.583058483348882, 21.507799635884947, 22.46085915528853, 23.419289399442857, 24.25565771744406, 25.00023546697811, 25.79852046401954, 26.622682589405425, 27.499575466889464, 28.416504656151726, 29.327086625254026]
[0.8295328501366263, 1.6082879389935532, 2.365071881142918, 3.124785309519958, 3.889381214943122, 4.682295487233811, 5.50450462180125, 6.356985114054664, 7.224601285151064, 8.07561703429064, 8.92126205867948, 9.782531009615917, 10.64477645596158, 11.496280452805768, 12.29554194525642, 13.093338694593236, 13.877952755905511, 14.673308266719264, 15.494540

[0.8861695838716754, 1.6473477553625526, 2.377766321462843, 3.122344070996895, 3.9635948660442213, 4.82876979861756, 5.803800464728709, 6.708035213671047, 7.57028066001671, 8.370518647876587, 9.150738479847352, 9.923634596248927, 10.715572373130392, 11.557799663586943, 12.505000210535181, 13.48979583073858, 14.477520937169654, 15.456945832622315, 16.36655130631539, 17.192178174815115, 17.983139456287358, 18.761894545144283, 19.55236757891191, 20.413148282143737, 21.295900132083123, 22.154239596791886, 23.01648504313755, 23.882148223415502, 24.806889375951563, 25.750672188967513, 26.676878084617414, 27.562071173079865, 28.381350821419627, 29.139111258978218]
[1.0780509317843854, 1.9329726625608612, 2.8328131821616864, 3.774154756654574, 4.692548689030674, 5.6133838599298365, 6.510783141007599, 7.342757229667288, 8.154713162437865, 8.964227856685378, 9.79669019304968, 10.668700593487593, 11.608577424866642, 12.569937155248642, 13.492237069261643, 14.350576533970406, 15.166438448377882, 1

[0.4042690994191445, 1.3163158116352822, 1.8651062316197247, 2.1390131939073336, 2.9211860166965478, 3.6926173899842873, 4.512873533733276, 5.525011025394975, 6.674346122052785, 7.7914568702061695, 8.841189435123031, 9.806943394846542, 10.739984758361016, 11.66081992926018, 12.596302531297717, 13.498584289421604, 14.404772029182393, 15.320724723035429, 16.273784242439017, 17.224890771024153, 18.13303150160339, 19.03140727809038, 19.948336467352643, 20.894072271187042, 21.834925597975317, 22.731836631348468, 23.63314189406313, 24.49245785418112, 25.330290915296157, 26.180330169026508, 27.02109271636922, 28.004911841163395, 28.93160598451791]
[0.6879410157990031, 1.6146351591535155, 2.455397706496229, 3.2732126117221556, 4.110557425132582, 4.99916824752732, 5.946368794475557, 6.925305442223606, 7.877876713922581, 8.85290738003373, 9.790342972889718, 10.68383627233058, 11.550475948017755, 12.417115623704932, 13.298890978235095, 14.201660984063595, 15.090271806458333, 15.953005500508608, 1

[0.4340522094005066, 1.4105476186254933, 2.339683000503068, 3.2390352723992812, 4.133505067249369, 5.045063531760894, 5.956621996272419, 6.8755041763531315, 7.835887411325906, 8.846560159873768, 9.836238257123291, 10.797109739800678, 11.768234424274928, 12.778418925118176, 13.7837209489153, 14.742151193069626, 15.687398749199412, 16.43246474643808, 17.5119804213363, 18.556342261502422, 19.328261882494775, 20.32770493383655, 21.330077471406, 22.9871901808608, 23.918766801261437, 24.905515412283286, 25.874198858234475, 26.788198561269063, 27.707080741349774, 28.63865736175041, 29.5331271566005]
[0.2714657237645463, 1.1376171517471096, 2.04624613003096, 3.053012896941921, 4.0954217462895945, 4.877106321374196, 5.843348528802321, 6.762718956587646, 7.6679302009392085, 8.611713013955159, 9.527665707808195, 10.441177163138171, 11.33955293962516, 12.242322945453658, 13.179270290605034, 14.169924883263782, 15.136655338396519, 16.08190289452631, 17.00273806542547, 17.930408704189208, 18.8766327

# Calculating RR intervals (RR)

In [51]:
def detectrr(rPeaks):
    
    #Obtaining RR intervals (difference in time between consecutive R-Peaks)
    rr = []
    for i in range(len(rPeaks) - 1):
        xx = rPeaks[i + 1] - rPeaks[i]
        rr.append(xx)
        i = i + 1
    return rr

In [52]:
rr_interval = detectrr(rPeaks[1])
print (rr_interval)

[0.5766205391473546, 0.5717380621012298, 0.5736910529196799, 0.5736910529196795, 0.5688085758735548, 0.848086262911901, 0.5526964016213425, 0.5497669153936675, 0.5570906309628549, 0.5590436217813046, 0.5082658605016057, 0.6107978784702297, 0.5775970345565797, 0.5834560070119288, 0.578573529965805, 0.5966386950364679, 0.6376515022239158, 0.6708523461375648, 0.4355169525143445, 0.6684111076145047, 0.6664581167960524, 0.6659698690914411, 0.6664581167960542, 0.6693876030237274, 0.42331075989903155, 0.6615756397499286, 1.097092592264275, 0.7123534010296275, 0.7333480523279654, 0.7841258136076625, 0.8632219417548868, 0.9628244734958393, 0.9833308770895606, 0.951594776289749, 0.9125349599207517, 0.8793341160071009, 0.8622454463456641, 0.9008170150100483, 0.870545657324076, 0.8324623363643013, 0.8505275014349643]


# HRV Calculation

https://pyhrv.readthedocs.io/en/latest/_pages/api/hrv.html

In [53]:
with open(hrv_csv_name, 'a', encoding='UTF8', newline='') as f:
    header = ['time','hr_mean', 'sdnn', 'rmssd', 'sdann']
    writer = csv.writer(f)
    # write the header
    writer.writerow(header)

In [54]:
#for n in range (len(split_values)):

for n in range (2):
    
    #Calculating HRV based on rPeaks calculated before
    hrv1 = hrv(rpeaks=(rPeaks[n]), sampling_rate=fs, show= False, plot_tachogram= False, plot_ecg= False)

    #Calculating HRV with filtered ecg, default parameters
    #hrv1 = hrv(signal=(split_values[n]), sampling_rate=fs,  show= False, plot_tachogram= False, plot_ecg= False)
    
    
    
    # the variable `hrv` is classbiosppy.utils.ReturnTuple(values, names=None), must be converted to ordered dictionary
    hrv_dic = hrv1.as_dict()

    mean_rr = hrv_dic["hr_mean"]
    sdnn = hrv_dic["sdnn"]
    rmssd = hrv_dic["rmssd"]
    sdann = hrv_dic["sdann"]
    
    col1 = (n + 1) * time_frame
        
    data = [col1, mean_rr, sdnn, rmssd, sdann]
        
    with open(hrv_csv_name, 'a', encoding='UTF8', newline='') as f:
        writer = csv.writer(f)
        
        # write the data
        writer.writerow(data)
    
    del hrv1 #cleaning the variable


/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/utils.py:368: UserWarning: Signal duration is to short for segmentation into 300000s. Input data will be returned.
  warnings.warn("Signal duration is to short for segmentation into %is. Input data will be returned." % duration)
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:303: UserWarning: Signal duration too short for SDANN computation.
  warnings.warn("Signal duration too short for SDANN computation.")
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:576: UserWarning: CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a malfunction of the function. This function will be reviewed over the next updates to solve this issue
  warnings.warn('CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a '
/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/nonlinear.py:286: UserWarning: Not enou

# a passar o tempo a funçao das rpeaks

Mean Heart Rate:  68.2253974404475
SDNN:  100.9520385964312
RMSSD:  81.6052919098206
SDANN:  17.108399636016706
